# EXP-006 — Thiên lệch chủng tộc trên RAF-DB
Tách RAF-DB test theo nhãn chủng tộc do người gán (`basic/Annotation/manual.zip`: 0 Caucasian · 1 African-American · 2 Asian), tính UAR/WAR zero-shot riêng từng nhóm.
- So sánh: **CLIP gốc** (không head) · **Paper** (checkpoint trong repo) · **EXP-000** (3 seed) · **EXP-004** λ = 1 ep30 (3 seed).
- Không train. Feature CLIP của ảnh/text tính **1 lần**, mỗi head chỉ là một phép nhân ma trận → chạy nhanh.
- Tiền xử lý giống `test.py`: `Resize((224, 224))` + `ToTensor()`, prompt trung bình theo lớp.

Nhóm Asian chỉ có 483 ảnh (Anger 9, Fear 12) → thêm **UAR5** (bỏ Anger, Fear) và bootstrap CI cho Δ = Asian − Caucasian.

Tiêu chí: **có thiên lệch** nếu UAR(Asian) thấp hơn UAR(Caucasian) ≥ 3 điểm, ổn định qua 3 seed, và CI 95% không chứa 0.

# Cấu hình

In [1]:
SEEDS = [1, 2, 3]
BRANCH = "exp/006-rafdb-race-bias"   # chỉ cần models/clip + models/Text.py (không cần lavis)
TEST_ZIP = "/content/drive/MyDrive/Datasets/static_test_sets.zip"
OUT_DIR = "/content/drive/MyDrive/ExpCLIP_output/exp006"
ATTR_CSV = f"{OUT_DIR}/rafdb_test_attributes.csv"   # trích từ basic/Annotation/manual.zip; không commit (giấy phép RAF-DB), tự upload
EXP000_ROOT = "/content/drive/MyDrive/ExpCLIP_output"  # tìm đệ quy expclip_l14_face_seed{N}-model.pth
EXP004_DIR = "/content/drive/MyDrive/ExpCLIP_output/exp004"
N_BOOT = 2000
MAIN_PROMPT = "Expression Ensemble Five"

PROMPT_NAMES = ["Class Name", "An Expression of Name", "A Photo of A Face with An Expression of Name",
                "Expression Ensemble Five", "Expression Ensemble Ten"]   # cùng thứ tự với FER_prompt_list
RACE = {0: "Caucasian", 1: "AfrAm", 2: "Asian"}
GENDER = {0: "Nam", 1: "Nữ", 2: "Không rõ"}
AGE = {0: "0-3", 1: "4-19", 2: "20-39", 3: "40-69", 4: "70+"}

# Mount Drive & GPU

In [2]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(OUT_DIR, exist_ok=True)
assert os.path.exists(ATTR_CSV), f"chưa upload {ATTR_CSV}"
!nvidia-smi --query-gpu=name,memory.total --format=csv

Mounted at /content/drive
name, memory.total [MiB]
Tesla T4, 15360 MiB


# Clone repo & giải nén test set

In [3]:
if not os.path.isdir("/content/Exp-CLIP"):
    !git clone -b $BRANCH https://github.com/huongng389/Exp-CLIP.git /content/Exp-CLIP
!cd /content/Exp-CLIP && git checkout $BRANCH && git pull && git log --oneline -1   # ghi lại commit vào research_log
!mkdir -p /content/static_test_sets
!unzip -q -n "$TEST_ZIP" -d /content/static_test_sets
!ls /content/static_test_sets/RAFDB/test
# CLIP vendored cần ftfy và pkg_resources (setuptools mới đã bỏ)
!pip install -q ftfy "setuptools<81"

Cloning into '/content/Exp-CLIP'...
remote: Enumerating objects: 427, done.
remote: Counting objects: 100% (427/427), done.
remote: Compressing objects: 100% (154/154), done.
remote: Total 427 (delta 312), reused 383 (delta 269), pack-reused 0 (from 0)
Receiving objects: 100% (427/427), 32.27 MiB | 20.34 MiB/s, done.
Resolving deltas: 100% (312/312), done.
Already on 'exp/006-rafdb-race-bias'
Your branch is up to date with 'origin/exp/006-rafdb-race-bias'.
Already up to date.
6a4b0ac (HEAD -> exp/006-rafdb-race-bias, origin/exp/006-rafdb-race-bias) EXP-005: add executed separate-text-head notebook (seed 1, sg/joint x mu 0/0.1)
0_Happiness  1_Sadness	2_Neutral  3_Anger  4_Surprise	5_Disgust  6_Fear
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.8 MB/s eta 0:00:00


# Tìm checkpoint
Thiếu file nào thì bỏ qua (in cảnh báo).

In [4]:
import glob

def find_ckpt(root, name):
    hits = glob.glob(f"{root}/**/{name}", recursive=True)
    return hits[0] if hits else None

HEADS = {
    "CLIP gốc": {0: None},
    "Paper": {0: "/content/Exp-CLIP/checkpoint/ExpCLIP_L14_model.pth"},
    "EXP-000": {s: find_ckpt(EXP000_ROOT, f"expclip_l14_face_seed{s}-model.pth") for s in SEEDS},
    "EXP-004": {s: f"{EXP004_DIR}/exp004_anchor_crossA_l1_l14_seed{s}-e30-model.pth" for s in SEEDS},
}
for m, d in HEADS.items():
    for s, p in list(d.items()):
        if p is not None and not os.path.exists(p):
            print(f"CẢNH BÁO: {m} seed {s}: không thấy checkpoint ({p}) → bỏ qua")
            del d[s]
        else:
            print(f"{m} seed {s}: {p}")

CLIP gốc seed 0: None
Paper seed 0: /content/Exp-CLIP/checkpoint/ExpCLIP_L14_model.pth
EXP-000 seed 1: /content/drive/MyDrive/ExpCLIP_output/expclip_l14_face_seed1-model.pth
EXP-000 seed 2: /content/drive/MyDrive/ExpCLIP_output/expclip_l14_face_seed2-model.pth
EXP-000 seed 3: /content/drive/MyDrive/ExpCLIP_output/expclip_l14_face_seed3-model.pth
EXP-004 seed 1: /content/drive/MyDrive/ExpCLIP_output/exp004/exp004_anchor_crossA_l1_l14_seed1-e30-model.pth
EXP-004 seed 2: /content/drive/MyDrive/ExpCLIP_output/exp004/exp004_anchor_crossA_l1_l14_seed2-e30-model.pth
EXP-004 seed 3: /content/drive/MyDrive/ExpCLIP_output/exp004/exp004_anchor_crossA_l1_l14_seed3-e30-model.pth


# Feature CLIP L/14 (ảnh + text), tính 1 lần

In [5]:
import sys, numpy as np, pandas as pd, torch, torch.nn.functional as F
import torchvision.datasets as datasets, torchvision.transforms as T
sys.path.insert(0, "/content/Exp-CLIP")
from models.clip import clip
from models.Text import FER_prompt_list

clip_model, _ = clip.load("ViT-L/14", "cuda")
clip_model.eval()
ds = datasets.ImageFolder("/content/static_test_sets/RAFDB/test",
                          T.Compose([T.Resize((224, 224)), T.ToTensor()]))   # giống test.py
loader = torch.utils.data.DataLoader(ds, batch_size=256, shuffle=False, num_workers=4)
with torch.no_grad():
    img = torch.cat([clip_model.encode_image(x.cuda()).float().cpu() for x, _ in loader])
    txt = {pn: clip_model.encode_text(clip.tokenize(pl["RAFDB"], context_length=77, truncate=True).cuda()).float().cpu()
           for pn, pl in zip(PROMPT_NAMES, FER_prompt_list)}
y = np.array(ds.targets)
classes = [c.split("_", 1)[1] for c in ds.classes]   # 0_Happiness → Happiness
print(img.shape, classes, {pn: tuple(t.shape) for pn, t in txt.items()})

/content/Exp-CLIP/models/clip/clip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging
100%|████████████████████████████████████████| 890M/890M [00:04<00:00, 209MiB/s]


torch.Size([3068, 768]) ['Happiness', 'Sadness', 'Neutral', 'Anger', 'Surprise', 'Disgust', 'Fear'] {'Class Name': (7, 768), 'An Expression of Name': (7, 768), 'A Photo of A Face with An Expression of Name': (7, 768), 'Expression Ensemble Five': (35, 768), 'Expression Ensemble Ten': (70, 768)}


# Ghép nhãn chủng tộc / giới tính / tuổi

In [6]:
attr = pd.read_csv(ATTR_CSV).set_index("image")
names = [os.path.basename(p) for p, _ in ds.samples]
assert len(names) == len(attr) == 3068 and set(names) == set(attr.index), "tên ảnh không khớp với CSV"
attr = attr.loc[names]
assert (attr["emotion"].values == np.array(classes)[y]).all(), "nhãn cảm xúc không khớp"
race = attr["race"].map(RACE).values
gender = attr["gender"].map(GENDER).values
age = attr["age"].map(AGE).values
print(pd.crosstab(race, np.array(classes)[y], margins=True))

col_0      Anger  Disgust  Fear  Happiness  Neutral  Sadness  Surprise   All
row_0                                                                       
AfrAm         10        7     5        101       49       38        24   234
Asian          9       27    12        196      126       91        22   483
Caucasian    143      126    57        888      505      349       283  2351
All          162      160    74       1185      680      478       329  3068


# Dự đoán & metric

In [7]:
def load_W(path):
    if path is None:
        return None
    sd = torch.load(path, map_location="cpu")
    return next(v for k, v in sd.items() if "mlp.weight" in k).float()   # (4096, 768)

def predict(W, pn):
    i, t = (img, txt[pn]) if W is None else (img @ W.T, txt[pn] @ W.T)
    i, t = F.normalize(i, dim=-1), F.normalize(t, dim=-1)
    return (i @ t.T).view(len(i), 7, -1).mean(-1).argmax(1).numpy()   # prompt xếp theo lớp, như test.py

def per_class(y_, p_):
    return np.array([(p_[y_ == c] == c).mean() * 100 if (y_ == c).any() else np.nan for c in range(7)])

KEEP5 = [c for c in range(7) if classes[c] not in ("Anger", "Fear")]
def metrics(y_, p_):
    acc = per_class(y_, p_)
    return {"UAR": np.nanmean(acc), "UAR5": np.nanmean(acc[KEEP5]), "WAR": (p_ == y_).mean() * 100,
            **{f"acc_{classes[c]}": acc[c] for c in range(7)}}

GROUPS = {"Tất cả": np.ones(len(y), bool), **{r: race == r for r in RACE.values()}}
preds, rows = {}, []
for m, d in HEADS.items():
    for s, path in d.items():
        W = load_W(path)
        for pn in PROMPT_NAMES:
            p = preds[(m, s, pn)] = predict(W, pn)
            for g, mask in GROUPS.items():
                rows.append({"model": m, "seed": s, "prompt": pn, "group": g, "n": int(mask.sum()), **metrics(y[mask], p[mask])})
res = pd.DataFrame(rows)
res.to_csv(f"{OUT_DIR}/exp006_race_bias-results.csv", index=False)
np.savez_compressed(f"{OUT_DIR}/exp006_race_bias-preds.npz", names=np.array(names), y=y, race=race,
                    **{f"{m}|{s}|{pn}": p for (m, s, pn), p in preds.items()})
print(len(res), "dòng →", OUT_DIR)

160 dòng → /content/drive/MyDrive/ExpCLIP_output/exp006


# Kiểm tra: số 'Tất cả' phải khớp `test.py`
So với kết quả đã ghi của EXP-000 / EXP-004 (RAFDB). Lệch ≤ ~0.1 là do GPU fp16; lệch lớn thì pipeline sai, dừng lại.

In [8]:
REF = {"EXP-000|1|Class Name": [50.36, 56.58], "EXP-000|1|An Expression of Name": [56.73, 64.05], "EXP-000|1|A Photo of A Face with An Expression of Name": [54.94, 65.12], "EXP-000|1|Expression Ensemble Five": [58.88, 66.88], "EXP-000|1|Expression Ensemble Ten": [57.73, 65.91], "EXP-000|2|Class Name": [49.61, 55.9], "EXP-000|2|An Expression of Name": [54.35, 62.26], "EXP-000|2|A Photo of A Face with An Expression of Name": [55.25, 64.44], "EXP-000|2|Expression Ensemble Five": [57.82, 65.78], "EXP-000|2|Expression Ensemble Ten": [56.65, 64.18], "EXP-000|3|Class Name": [50.73, 51.21], "EXP-000|3|An Expression of Name": [54.94, 59.68], "EXP-000|3|A Photo of A Face with An Expression of Name": [56.74, 64.77], "EXP-000|3|Expression Ensemble Five": [57.5, 63.3], "EXP-000|3|Expression Ensemble Ten": [57.19, 63.72], "EXP-004|1|Class Name": [54.65, 57.82], "EXP-004|1|An Expression of Name": [57.84, 62.68], "EXP-004|1|A Photo of A Face with An Expression of Name": [58.55, 68.68], "EXP-004|1|Expression Ensemble Five": [58.73, 65.94], "EXP-004|1|Expression Ensemble Ten": [58.86, 66.85], "EXP-004|2|Class Name": [53.74, 57.89], "EXP-004|2|An Expression of Name": [56.11, 61.8], "EXP-004|2|A Photo of A Face with An Expression of Name": [57.96, 67.89], "EXP-004|2|Expression Ensemble Five": [57.07, 64.77], "EXP-004|2|Expression Ensemble Ten": [57.95, 65.84], "EXP-004|3|Class Name": [52.44, 54.86], "EXP-004|3|An Expression of Name": [56.63, 61.86], "EXP-004|3|A Photo of A Face with An Expression of Name": [59.81, 68.29], "EXP-004|3|Expression Ensemble Five": [58.26, 65.06], "EXP-004|3|Expression Ensemble Ten": [58.5, 66.0]}

chk = []
for k, (u, w) in REF.items():
    m, s, pn = k.split("|")
    r = res[(res.model == m) & (res.seed == int(s)) & (res.prompt == pn) & (res.group == "Tất cả")]
    if len(r):
        chk.append({"key": k, "dUAR": r.UAR.item() - u, "dWAR": r.WAR.item() - w})
chk = pd.DataFrame(chk)
print(f"{len(chk)}/{len(REF)} cấu hình so được | max |ΔUAR| = {chk.dUAR.abs().max():.2f}, max |ΔWAR| = {chk.dWAR.abs().max():.2f}")
print(chk.round(2).sort_values("dUAR", key=abs, ascending=False).head(5).to_string(index=False))

30/30 cấu hình so được | max |ΔUAR| = 0.21, max |ΔWAR| = 0.14
                                                   key  dUAR  dWAR
                       EXP-000|3|An Expression of Name -0.21 -0.06
                                  EXP-000|1|Class Name  0.19  0.04
                     EXP-004|3|Expression Ensemble Ten  0.15 -0.03
                    EXP-000|2|Expression Ensemble Five  0.13  0.06
EXP-000|3|A Photo of A Face with An Expression of Name  0.13  0.09


# Bảng 1 — Ensemble Five: UAR / UAR5 / WAR theo nhóm (mean ± std qua seed)

In [9]:
pd.set_option("display.width", 250)
def fmt(x):
    return f"{x.mean():.2f}" if len(x) == 1 else f"{x.mean():.2f} ± {x.std(ddof=1):.2f}"

main = res[res.prompt == MAIN_PROMPT]
for metric in ["UAR", "UAR5", "WAR"]:
    t = main.pivot_table(index="model", columns="group", values=metric, aggfunc=fmt)[["Tất cả", *RACE.values()]]
    d = main.pivot_table(index=["model", "seed"], columns="group", values=metric)
    t["Δ Asian−Cauc"] = (d["Asian"] - d["Caucasian"]).groupby("model").apply(fmt)
    print(f"--- {metric}\n{t.loc[list(HEADS)].to_string()}\n")

--- UAR
group           Tất cả     Caucasian         AfrAm         Asian Δ Asian−Cauc
model                                                                        
CLIP gốc         52.10         51.98         50.95         53.35         1.37
Paper            57.95         57.13         60.34         60.85         3.72
EXP-000   58.11 ± 0.73  58.11 ± 1.04  57.42 ± 1.15  58.60 ± 1.67  0.48 ± 2.69
EXP-004   57.97 ± 0.91  57.95 ± 0.79  58.37 ± 1.30  58.67 ± 1.07  0.72 ± 0.28

--- UAR5
group           Tất cả     Caucasian         AfrAm         Asian Δ Asian−Cauc
model                                                                        
CLIP gốc         46.42         45.82         41.33         51.91         6.09
Paper            56.30         54.99         52.48         64.08         9.09
EXP-000   57.73 ± 1.29  57.42 ± 1.29  54.39 ± 1.24  60.92 ± 3.00  3.50 ± 3.06
EXP-004   57.48 ± 1.10  57.59 ± 0.79  52.39 ± 2.28  59.54 ± 2.09  1.95 ± 1.50

--- WAR
group           Tất cả     Caucasian 

# Bảng 2 — Δ UAR (Asian − Caucasian) theo 5 kiểu prompt
Thiên lệch có ổn định khi đổi prompt không?

In [10]:
d = res.pivot_table(index=["model", "seed", "prompt"], columns="group", values="UAR")
delta = (d["Asian"] - d["Caucasian"]).groupby(["model", "prompt"]).mean().unstack()[PROMPT_NAMES]
print(delta.loc[list(HEADS)].round(2).to_string())

prompt    Class Name  An Expression of Name  A Photo of A Face with An Expression of Name  Expression Ensemble Five  Expression Ensemble Ten
model                                                                                                                                       
CLIP gốc       -1.05                   3.55                                         -0.45                      1.37                    -1.05
Paper          -6.57                   3.27                                         -1.98                      3.72                     2.00
EXP-000        -5.65                  -0.70                                         -0.64                      0.48                     0.69
EXP-004        -1.80                   2.53                                         -2.45                      0.72                    -0.58


# Bảng 3 — Accuracy từng lớp: Asian vs Caucasian (Ensemble Five, mean seed)
Lớp nào gây ra chênh lệch? (n nhỏ ở Anger/Fear → đọc thận trọng)

In [11]:
cols = [f"acc_{c}" for c in classes]
n_cls = pd.crosstab(race, np.array(classes)[y])[classes]
print("Số ảnh:\n" + n_cls.loc[list(RACE.values())].to_string() + "\n")
for m in HEADS:
    t = main[(main.model == m) & main.group.isin(["Caucasian", "Asian"])].groupby("group")[cols].mean()
    t.columns = classes
    t.loc["Δ"] = t.loc["Asian"] - t.loc["Caucasian"]
    print(f"--- {m}\n{t.round(1).to_string()}\n")

Số ảnh:
col_0      Happiness  Sadness  Neutral  Anger  Surprise  Disgust  Fear
row_0                                                                 
Caucasian        888      349      505    143       283      126    57
AfrAm            101       38       49     10        24        7     5
Asian            196       91      126      9        22       27    12

--- CLIP gốc
           Happiness  Sadness  Neutral  Anger  Surprise  Disgust  Fear
group                                                                 
Asian           19.9     64.8     32.5   88.9      68.2     74.1  25.0
Caucasian       16.0     49.3     43.2   90.9      48.4     72.2  43.9
Δ                3.9     15.6    -10.6   -2.0      19.8      1.9 -18.9

--- Paper
           Happiness  Sadness  Neutral  Anger  Surprise  Disgust  Fear
group                                                                 
Asian           94.4     67.0     15.9   88.9      72.7     70.4  16.7
Caucasian       91.2     51.3     18.4   88.

# Bootstrap CI 95% cho Δ = Asian − Caucasian (Ensemble Five)
Lấy mẫu lại ảnh trong từng nhóm (cùng chỉ số cho mọi seed), Δ = trung bình qua seed.

In [12]:
rng = np.random.default_rng(0)
ia, ic = np.where(race == "Asian")[0], np.where(race == "Caucasian")[0]
boot = []
for m, d in HEADS.items():
    P = [preds[(m, s, MAIN_PROMPT)] for s in d]
    draws = {"UAR": [], "UAR5": [], "WAR": []}
    for _ in range(N_BOOT):
        a, c = rng.choice(ia, len(ia)), rng.choice(ic, len(ic))
        ms = [(metrics(y[a], p[a]), metrics(y[c], p[c])) for p in P]
        for k in draws:
            draws[k].append(np.mean([ma[k] - mc[k] for ma, mc in ms]))
    for k, v in draws.items():
        point = np.mean([metrics(y[ia], p[ia])[k] - metrics(y[ic], p[ic])[k] for p in P])
        lo, hi = np.percentile(v, [2.5, 97.5])
        boot.append({"model": m, "metric": k, "Δ": point, "CI 2.5%": lo, "CI 97.5%": hi, "CI chứa 0": lo <= 0 <= hi})
boot = pd.DataFrame(boot)
boot.to_csv(f"{OUT_DIR}/exp006_race_bias-bootstrap.csv", index=False)
print(boot.round(2).to_string(index=False))

   model metric     Δ  CI 2.5%  CI 97.5%  CI chứa 0
CLIP gốc    UAR  1.37    -5.66      8.50       True
CLIP gốc   UAR5  6.09    -0.19     12.38       True
CLIP gốc    WAR -0.62    -5.43      4.17       True
   Paper    UAR  3.72    -2.87     10.26       True
   Paper   UAR5  9.09     2.64     15.20      False
   Paper    WAR  1.99    -2.70      6.61       True
 EXP-000    UAR  0.48    -5.72      6.42       True
 EXP-000   UAR5  3.50    -2.75      9.41       True
 EXP-000    WAR -0.30    -4.38      3.73       True
 EXP-004    UAR  0.72    -4.60      6.18       True
 EXP-004   UAR5  1.95    -3.88      7.74       True
 EXP-004    WAR -2.61    -6.87      1.67       True


# Phụ — Giới tính và tuổi (Ensemble Five, UAR, mean seed)
Cùng code, gần như không tốn thêm công. Nhóm < 100 ảnh chỉ để tham khảo.

In [13]:
for name, attr_ in [("Giới tính", gender), ("Tuổi", age)]:
    out = {}
    for m, d in HEADS.items():
        out[m] = {g: np.mean([metrics(y[attr_ == g], preds[(m, s, MAIN_PROMPT)][attr_ == g])["UAR"] for s in d])
                  for g in pd.unique(attr_)}
    t = pd.DataFrame(out).T
    t.loc["n"] = pd.Series(attr_).value_counts()
    print(f"--- {name}\n{t.round(2).to_string()}\n")

--- Giới tính
               Nữ      Nam  Không rõ
CLIP gốc    52.81    48.74     60.33
Paper       60.60    53.30     65.43
EXP-000     59.62    54.37     67.59
EXP-004     59.04    55.20     67.27
n         1620.00  1249.00    199.00

--- Tuổi
            4-19    20-39   40-69    70+     0-3
CLIP gốc   49.06    52.96   41.65  31.80   53.70
Paper      52.10    58.62   51.66  56.13   58.67
EXP-000    50.40    59.07   52.43  48.04   58.88
EXP-004    50.26    59.54   51.62  43.01   56.24
n         486.00  1662.00  502.00  89.00  329.00

